# N18: SOTA Family Expansion (TabICLv2 + TabM)

**Question:** N17 executed and confirmed the double-correction fix (TabPFN OOF restored to exactly
**0.94872**, fold-for-fold identical to N14v3) plus two new levers: a searched blend weight
(re-found alpha=0.28) and OOF-learned per-class thresholds (+0.00007). Result: **new organic best,
Public LB 0.95034** (OOF 0.95038). The N16 dilution hypothesis was **REJECTED** — the 3x3
seed-ensemble (0.94998) beat every solo GBDT, including HGBC-solo at 0.93491, which also discredits
Era-1's "v0.7 HGBC-solo 0.95020" as a trustworthy comparator on this exact feature matrix.

**Falsifiable hypothesis:** An exhaustive SOTA research sweep (TabArena NeurIPS 2025, TabICL ICML
2025 / v2 Feb 2026, Chunked TabPFN, multiclass threshold-tuning literature) identified two never-
tried, orthogonal information sources:
- **TabICLv2** — a tabular foundation model with column-then-row attention that scales to 500K+
  rows via CPU/disk offloading, and beats both TabPFNv2 and CatBoost on >10K-row datasets. This
  removes TabPFN's structural handicap: TabPFN only ever sees a 100K subsample (14.5%) of each
  552K-row fold-train set; TabICL can see the **full fold-train set**.
- **TabM** — TabArena's #1-ranked model after post-hoc ensembling (above LightGBM, RealMLP,
  CatBoost). A parameter-efficient ensemble of k=32 MLPs trained in parallel, pure PyTorch.

If either family scores >=0.949 OOF, the N-way blend search should push past N17's 0.95038 OOF.
If both underperform, the search will assign them zero weight and the result falls back to N17's
proven blend — no downside beyond compute.

**Not this notebook:** RealMLP (dropped, see N15v4/N17). N12-style anchor CSV probing (non-organic).

Attach `prior-labsai/tabpfn-3`. GPU T4 x2. **Internet ON** (required for `pip install tabicl tabm`
and their checkpoint downloads).

In [ ]:
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
os.environ["TABPFN_NO_BROWSER"] = "1"
os.environ["TABPFN_MAX_BATCHED_TEST_ROWS"] = "4096"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q tabpfn 2>/dev/null || true
!pip install -q tabicl 2>/dev/null || true
!pip install -q tabm 2>/dev/null || true

In [ ]:
import os, gc, time, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
warnings.filterwarnings("ignore")

ID_COL, TARGET_COL = "id", "health_condition"
N_FOLDS, SEED, N_CLASSES = 5, 42, 3
GPU_ENABLED = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if GPU_ENABLED else 0
DEVICE = "cuda" if GPU_ENABLED else "cpu"
cb_task = "GPU" if GPU_ENABLED else "CPU"
print(f"GPU={GPU_ENABLED} | n_gpus={N_GPUS}")

_KAGGLE = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
TABPFN_CKPT = f"{_KAGGLE}/tabpfn-v3-classifier-v3_default.ckpt"
os.environ["TABPFN_MODEL_CACHE_DIR"] = _KAGGLE
os.environ["TABPFN_NO_BROWSER"] = "1"

TABPFN_OK = False
if os.path.isfile(TABPFN_CKPT):
    try:
        from tabpfn import TabPFNClassifier
        TABPFN_OK = True
        print(f"TabPFN checkpoint OK: {TABPFN_CKPT}")
    except Exception as e:
        print(f"TabPFN import failed: {e}")
else:
    print(f"TabPFN checkpoint MISSING: {TABPFN_CKPT}")

TABICL_OK = False
try:
    from tabicl import TabICLClassifier
    TABICL_OK = True
    print("TabICL import OK")
except Exception as e:
    print(f"TabICL unavailable: {e}")

TABM_OK = False
try:
    import tabm
    TABM_OK = True
    print("TabM import OK")
except Exception as e:
    print(f"TabM unavailable: {e}")

# NOTE: no prior_correct() here. TabPFNClassifier already gets balance_probabilities=True
# below; manually dividing by class priors on top of that is the N15v4 double-correction
# bug that collapsed TabPFN OOF from 0.94872 to 0.93302 (see N07 "double correction" trap,
# Rules.md Rule 21). Confirmed fixed and re-validated in N17.

def batch_predict(clf, X, bs=4096):
    X = np.asarray(X)
    return np.vstack([clf.predict_proba(X[i:i+bs]) for i in range(0, len(X), bs)])

In [ ]:
for p in ["/kaggle/input/competitions/playground-series-s6e7","/kaggle/input/playground-series-s6e7","../../data"]:
    if os.path.exists(os.path.join(p,"train.csv")):
        DATA_DIR=p; break
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
le=LabelEncoder(); y=le.fit_transform(train_df[TARGET_COL])
class_priors = np.bincount(y, minlength=N_CLASSES).astype(np.float64) / len(y)
print(f"Class priors: {class_priors}")

cat_cols=["stress_level","physical_activity_level","diet_type","gender","smoking_alcohol","sleep_quality"]
num_cols=["sleep_duration","heart_rate","bmi","calorie_expenditure","step_count","exercise_duration","water_intake"]
all_cols=cat_cols+num_cols
X_str=train_df[all_cols].fillna("__NA__").astype(str)
X_str_te=test_df[all_cols].fillna("__NA__").astype(str)
imp=SimpleImputer(strategy="median")
X_num=imp.fit_transform(train_df[num_cols]); X_num_te=imp.transform(test_df[num_cols])
te=TargetEncoder(cv=5,target_type="multiclass",random_state=SEED)
X_te=te.fit_transform(X_str,y); X_te_test=te.transform(X_str_te)
X_full=np.hstack([X_num,X_te]); X_full_test=np.hstack([X_num_te,X_te_test])
print(f"TE matrix: {X_full.shape}")

# Compact matrix (numeric + ordinal cat codes) shared by TabPFN (Family B) and TabICL (Family D).
# Built unconditionally so either family can use it regardless of the other's availability.
X_tab=X_num.copy(); X_tab_te=X_num_te.copy()
for c in cat_cols:
    cats=pd.Categorical(train_df[c].fillna("Missing").astype(str))
    X_tab=np.hstack([X_tab, cats.codes.astype(np.float64).reshape(-1,1)])
    te_codes=pd.Categorical(test_df[c].fillna("Missing").astype(str), categories=cats.categories).codes.astype(np.float64).reshape(-1,1)
    X_tab_te=np.hstack([X_tab_te, te_codes])
print(f"Compact matrix: {X_tab.shape}")

folds=list(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED).split(X_full,y))

In [ ]:
print("="*60+"\nFAMILY A: GBDT ENSEMBLE + NUMERIC TE (+ folded-in N16 solo dilution test)\n"+"="*60)
t0=time.time()
oof_g=np.zeros((len(train_df),N_CLASSES)); tst_g=np.zeros((len(test_df),N_CLASSES))
oof_solo_cb=np.zeros((len(train_df),N_CLASSES)); tst_solo_cb=np.zeros((len(test_df),N_CLASSES))
oof_solo_lgb=np.zeros((len(train_df),N_CLASSES)); tst_solo_lgb=np.zeros((len(test_df),N_CLASSES))
oof_solo_hgbc=np.zeros((len(train_df),N_CLASSES)); tst_solo_hgbc=np.zeros((len(test_df),N_CLASSES))
SEEDS=[42,2026,999]
for fold,(tr,va) in enumerate(folds):
    print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
    X_tr,X_va=X_full[tr],X_full[va]; y_tr=y[tr]
    sw=compute_sample_weight("balanced",y_tr)
    cc=np.bincount(y_tr,minlength=N_CLASSES); cb_w=[len(y_tr)/(N_CLASSES*c) for c in cc]
    fv=np.zeros((len(va),N_CLASSES)); ft=np.zeros((len(test_df),N_CLASSES)); nm=3*len(SEEDS)
    for s in SEEDS:
        m=CatBoostClassifier(iterations=1200,learning_rate=0.03,depth=6,class_weights=cb_w,random_seed=s,verbose=0,task_type=cb_task)
        m.fit(X_tr,y_tr,eval_set=(X_va,y[va]),early_stopping_rounds=100)
        cb_p_va=m.predict_proba(X_va); cb_p_te=m.predict_proba(X_full_test)
        fv+=cb_p_va/nm; ft+=cb_p_te/nm
        if s==SEED:
            oof_solo_cb[va]=cb_p_va; tst_solo_cb+=cb_p_te/N_FOLDS
        del m
        lp=dict(n_estimators=1200,learning_rate=0.03,num_leaves=63,class_weight="balanced",random_state=s,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        m=lgb.LGBMClassifier(**lp)
        m.fit(X_tr,y_tr,eval_set=[(X_va,y[va])],callbacks=[lgb.early_stopping(100,verbose=False)])
        lgb_p_va=m.predict_proba(X_va); lgb_p_te=m.predict_proba(X_full_test)
        fv+=lgb_p_va/nm; ft+=lgb_p_te/nm
        if s==SEED:
            oof_solo_lgb[va]=lgb_p_va; tst_solo_lgb+=lgb_p_te/N_FOLDS
        del m
        m=HistGradientBoostingClassifier(max_iter=1200,learning_rate=0.03,max_leaf_nodes=63,class_weight="balanced",random_state=s,early_stopping=True,validation_fraction=0.1,n_iter_no_change=100)
        m.fit(X_tr,y_tr,sample_weight=sw)
        hgbc_p_va=m.predict_proba(X_va); hgbc_p_te=m.predict_proba(X_full_test)
        fv+=hgbc_p_va/nm; ft+=hgbc_p_te/nm
        if s==SEED:
            oof_solo_hgbc[va]=hgbc_p_va; tst_solo_hgbc+=hgbc_p_te/N_FOLDS
        del m
    oof_g[va]=fv; tst_g+=ft/N_FOLDS
    print(f"BAcc={balanced_accuracy_score(y[va],fv.argmax(1)):.5f}"); gc.collect()
score_g=balanced_accuracy_score(y,oof_g.argmax(1))
score_solo_cb=balanced_accuracy_score(y,oof_solo_cb.argmax(1))
score_solo_lgb=balanced_accuracy_score(y,oof_solo_lgb.argmax(1))
score_solo_hgbc=balanced_accuracy_score(y,oof_solo_hgbc.argmax(1))
print(f">>> GBDT 3x3 ENSEMBLE OOF: {score_g:.5f} ({time.time()-t0:.0f}s) <<<")
print(f"    solo CatBoost   OOF: {score_solo_cb:.5f}")
print(f"    solo LightGBM   OOF: {score_solo_lgb:.5f}")
print(f"    solo HGBC       OOF: {score_solo_hgbc:.5f}")

solo_candidates = {"CatBoost-solo": (score_solo_cb, oof_solo_cb, tst_solo_cb),
                    "LightGBM-solo": (score_solo_lgb, oof_solo_lgb, tst_solo_lgb),
                    "HGBC-solo": (score_solo_hgbc, oof_solo_hgbc, tst_solo_hgbc)}
best_solo_name = max(solo_candidates, key=lambda k: solo_candidates[k][0])
best_solo_score = solo_candidates[best_solo_name][0]
DILUTION_CONFIRMED = best_solo_score > score_g
print(f"Dilution check: best solo={best_solo_name} ({best_solo_score:.5f}) vs 3x3 ensemble ({score_g:.5f}) -> "
      f"{'SOLO WINS (dilution confirmed)' if DILUTION_CONFIRMED else 'ENSEMBLE WINS (dilution hypothesis rejected)'}")

if DILUTION_CONFIRMED:
    base_name, base_oof, base_tst = f"GBDT {best_solo_name}", solo_candidates[best_solo_name][1], solo_candidates[best_solo_name][2]
else:
    base_name, base_oof, base_tst = "GBDT 3x3 Ensemble", oof_g, tst_g
score_base = balanced_accuracy_score(y, base_oof.argmax(1))
print(f">>> BASE GBDT SIGNAL SELECTED: {base_name} | OOF {score_base:.5f} <<<")

In [ ]:
print("="*60+"\nFAMILY B: TabPFN-3 compact 100K (balance_probabilities only)\n"+"="*60)
oof_t=np.zeros((len(train_df),N_CLASSES)); tst_t=np.zeros((len(test_df),N_CLASSES)); score_t=0.0
if TABPFN_OK:
    devices=["cuda:0","cuda:1"] if N_GPUS>=2 else DEVICE
    try:
        t0=time.time()
        for fold,(tr,va) in enumerate(folds):
            print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
            sub,_=train_test_split(np.arange(len(tr)), train_size=min(100_000,len(tr)), stratify=y[tr], random_state=SEED+fold)
            clf=TabPFNClassifier(device=devices,n_estimators=2,balance_probabilities=True,fit_mode="fit_with_cache",model_path=TABPFN_CKPT,ignore_pretraining_limits=True)
            clf.fit(X_tab[tr[sub]], y[tr[sub]])
            va_p=batch_predict(clf,X_tab[va]); te_p=batch_predict(clf,X_tab_te)
            oof_t[va]=va_p; tst_t+=te_p/N_FOLDS
            print(f"BAcc={balanced_accuracy_score(y[va], va_p.argmax(1)):.5f}")
            del clf; gc.collect()
            if GPU_ENABLED: torch.cuda.empty_cache()
        score_t=balanced_accuracy_score(y, oof_t.argmax(1))
        print(f">>> TabPFN OOF: {score_t:.5f} ({time.time()-t0:.0f}s) <<<")
    except Exception as e:
        print(f"TabPFN FAILED: {e}"); score_t=0.0
else:
    print("TabPFN skipped")

In [ ]:
print("="*60+"\nFAMILY D: TabICLv2 (FULL fold-train context, not subsampled)\n"+"="*60)
oof_i=np.zeros((len(train_df),N_CLASSES)); tst_i=np.zeros((len(test_df),N_CLASSES)); score_i=0.0
if TABICL_OK:
    try:
        t0=time.time()
        for fold,(tr,va) in enumerate(folds):
            print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
            fit_idx=tr
            try:
                clf=TabICLClassifier(device=DEVICE, offload_mode="auto", random_state=SEED+fold)
                clf.fit(X_tab[fit_idx], y[fit_idx])
            except Exception as e1:
                print(f"full-context fit failed ({e1}); falling back to 200K subsample")
                sub,_=train_test_split(np.arange(len(tr)), train_size=min(200_000,len(tr)), stratify=y[tr], random_state=SEED+fold)
                fit_idx=tr[sub]
                clf=TabICLClassifier(device=DEVICE, offload_mode="auto", random_state=SEED+fold)
                clf.fit(X_tab[fit_idx], y[fit_idx])
            va_p=batch_predict(clf,X_tab[va],bs=20000); te_p=batch_predict(clf,X_tab_te,bs=20000)
            oof_i[va]=va_p; tst_i+=te_p/N_FOLDS
            print(f"BAcc={balanced_accuracy_score(y[va], va_p.argmax(1)):.5f} (context={len(fit_idx)})")
            del clf; gc.collect()
            if GPU_ENABLED: torch.cuda.empty_cache()
        score_i=balanced_accuracy_score(y, oof_i.argmax(1))
        print(f">>> TabICL OOF: {score_i:.5f} ({time.time()-t0:.0f}s) <<<")
    except Exception as e:
        print(f"TabICL FAILED completely: {e}"); score_i=0.0
else:
    print("TabICL skipped (import failed)")

In [ ]:
print("="*60+"\nFAMILY E: TabM (k=32 parameter-efficient MLP mini-ensemble)\n"+"="*60)
oof_m=np.zeros((len(train_df),N_CLASSES)); tst_m=np.zeros((len(test_df),N_CLASSES)); score_m=0.0

def train_tabm_fold(X_tr, y_tr, X_va, y_va, X_te, seed, k=32, d_block=256, n_blocks=3,
                     max_epochs=50, patience=8, batch_size=4096, lr=2e-3, wd=1e-4):
    torch.manual_seed(seed)
    d_in = X_tr.shape[1]
    backbone = tabm.MLPBackbone(d_in=d_in, n_blocks=n_blocks, d_block=d_block, dropout=0.1)
    tabm.ensemble_linear_layers_(backbone, k=k)
    model = nn.Sequential(tabm.EnsembleView(k=k), backbone, tabm.LinearEnsemble(d_block, N_CLASSES, k=k)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_tr)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEVICE))

    ds = torch.utils.data.TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)
    Xva_t = torch.tensor(X_va, dtype=torch.float32, device=DEVICE)

    best_bacc, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            B, kk, C = logits.shape
            y_rep = yb.unsqueeze(1).expand(-1, kk).reshape(-1)
            loss = crit(logits.reshape(-1, C), y_rep)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va_proba = torch.softmax(model(Xva_t), dim=-1).mean(dim=1).cpu().numpy()
        bacc = balanced_accuracy_score(y_va, va_proba.argmax(1))
        if bacc > best_bacc:
            best_bacc = bacc; best_state = {kk2: v.clone() for kk2, v in model.state_dict().items()}; bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_proba = torch.softmax(model(Xva_t), dim=-1).mean(dim=1).cpu().numpy()
        te_chunks = []
        Xte_full = torch.tensor(X_te, dtype=torch.float32)
        for i in range(0, len(X_te), 20000):
            chunk = Xte_full[i:i+20000].to(DEVICE)
            te_chunks.append(torch.softmax(model(chunk), dim=-1).mean(dim=1).cpu().numpy())
        te_proba = np.vstack(te_chunks)
    del model, Xva_t
    return va_proba, te_proba, best_bacc

if TABM_OK:
    try:
        t0=time.time()
        for fold,(tr,va) in enumerate(folds):
            print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
            va_p, te_p, bacc = train_tabm_fold(X_full[tr], y[tr], X_full[va], y[va], X_full_test, seed=SEED+fold)
            oof_m[va]=va_p; tst_m+=te_p/N_FOLDS
            print(f"BAcc={bacc:.5f}")
            gc.collect()
            if GPU_ENABLED: torch.cuda.empty_cache()
        score_m=balanced_accuracy_score(y, oof_m.argmax(1))
        print(f">>> TabM OOF: {score_m:.5f} ({time.time()-t0:.0f}s) <<<")
    except Exception as e:
        print(f"TabM FAILED: {e}"); score_m=0.0
else:
    print("TabM skipped (import failed)")

In [ ]:
print("="*60+"\nFUSION: N-way weight search (Lever 1, with N17 pairwise guardrail floor)\n"+"="*60)

avail_names, avail_oofs, avail_tsts, avail_scores = [], [], [], []
if score_t>0.90: avail_names.append("TabPFN"); avail_oofs.append(oof_t); avail_tsts.append(tst_t); avail_scores.append(score_t)
if score_i>0.90: avail_names.append("TabICL"); avail_oofs.append(oof_i); avail_tsts.append(tst_i); avail_scores.append(score_i)
if score_m>0.90: avail_names.append("TabM"); avail_oofs.append(oof_m); avail_tsts.append(tst_m); avail_scores.append(score_m)
avail_names.append(base_name); avail_oofs.append(base_oof); avail_tsts.append(base_tst); avail_scores.append(score_base)
print("Available families:", dict(zip(avail_names, [round(s,5) for s in avail_scores])))

# Guardrail floor: N17's proven pairwise GBDT+TabPFN scan. This is the score N18 must beat
# to justify the added complexity/compute of TabICL and TabM.
best_score, best_name, best_oof, best_probs = score_base, f"{base_name} solo", base_oof, base_tst
if score_t>0.90:
    for a in np.arange(0.0,0.61,0.02):
        blend_oof=a*oof_t+(1-a)*base_oof
        sc=balanced_accuracy_score(y, blend_oof.argmax(1))
        if sc>best_score:
            best_score=sc; best_name=f"Pairwise TabPFN={a:.2f}+{base_name}={1-a:.2f}"
            best_oof=blend_oof; best_probs=a*tst_t+(1-a)*base_tst
print(f"Guardrail floor (N17-style pairwise): {best_name} | OOF {best_score:.5f}")

# N-way search over ALL available families (GBDT + any of TabPFN/TabICL/TabM that trained OK).
if len(avail_oofs) >= 2:
    def neg_bacc(w):
        w=np.abs(w); w=w/w.sum()
        blend=sum(w[i]*avail_oofs[i] for i in range(len(w)))
        return -balanced_accuracy_score(y, blend.argmax(1))
    res=minimize(neg_bacc, x0=np.ones(len(avail_oofs))/len(avail_oofs), method="Nelder-Mead", options={"maxiter":4000})
    nway_w=np.abs(res.x); nway_w/=nway_w.sum()
    nway_oof=sum(nway_w[i]*avail_oofs[i] for i in range(len(nway_w)))
    nway_tst=sum(nway_w[i]*avail_tsts[i] for i in range(len(nway_w)))
    nway_score=balanced_accuracy_score(y, nway_oof.argmax(1))
    print(f"N-way blend: {dict(zip(avail_names,[round(w,3) for w in nway_w]))} | OOF {nway_score:.5f}")
    if nway_score>best_score:
        best_score=nway_score; best_name=f"N-way{avail_names}"; best_oof=nway_oof; best_probs=nway_tst
        print("N-way blend ACCEPTED (beats N17 guardrail floor)")
    else:
        print("N-way blend REJECTED (guardrail floor stands -- new families added no lift)")

print(f">>> SELECTED BLEND: {best_name} | OOF {best_score:.5f} <<<")

In [ ]:
print("="*60+"\nOOF-LEARNED PER-CLASS THRESHOLDS (Lever 2)\n"+"="*60)

def neg_bacc_thresh(t):
    t=np.abs(t)
    scaled=best_oof*t
    return -balanced_accuracy_score(y, scaled.argmax(1))

res=minimize(neg_bacc_thresh, x0=np.ones(N_CLASSES), method="Nelder-Mead", options={"maxiter":2000,"xatol":1e-4,"fatol":1e-6})
learned_t=np.abs(res.x); learned_t/=learned_t.mean()
thresh_oof_raw=best_oof*learned_t
thresh_oof=thresh_oof_raw/thresh_oof_raw.sum(axis=1,keepdims=True)
thresh_score=balanced_accuracy_score(y, thresh_oof.argmax(1))
print(f"Learned thresholds: {learned_t} | OOF with thresholds: {thresh_score:.5f} (vs {best_score:.5f} without)")

if thresh_score>best_score:
    best_score=thresh_score; best_name+=" +thresh"; best_oof=thresh_oof
    scaled_tst=best_probs*learned_t
    best_probs=scaled_tst/scaled_tst.sum(axis=1,keepdims=True)
    print("Thresholds ACCEPTED (improved OOF)")
else:
    print("Thresholds REJECTED (no OOF improvement) -- guardrail kept identity weighting")
print(f">>> POST-THRESHOLD: {best_name} | OOF {best_score:.5f} <<<")

In [ ]:
PSEUDO_THRESHOLD=0.99
mask=best_probs.max(1)>=PSEUDO_THRESHOLD
n_pseudo=int(mask.sum()); print(f"Pseudo-labels: {n_pseudo} ({100*n_pseudo/len(test_df):.1f}%)")
if n_pseudo>1000:
    py=best_probs[mask].argmax(1)
    aug_X=np.vstack([X_full,X_full_test[mask]]); aug_y=np.concatenate([y,py])
    aug_tst=np.zeros((len(test_df),N_CLASSES))
    for fold,(tr,va) in enumerate(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED*2).split(aug_X,aug_y)):
        sw=compute_sample_weight("balanced",aug_y[tr])
        cc=np.bincount(aug_y[tr],minlength=N_CLASSES); cw=[len(aug_y[tr])/(N_CLASSES*c) for c in cc]
        fp=np.zeros((len(test_df),N_CLASSES))
        m=CatBoostClassifier(iterations=1200,learning_rate=0.03,depth=6,class_weights=cw,random_seed=SEED,verbose=0,task_type=cb_task)
        m.fit(aug_X[tr],aug_y[tr],eval_set=(aug_X[va],aug_y[va]),early_stopping_rounds=100)
        fp+=m.predict_proba(X_full_test)/3; del m
        lp=dict(n_estimators=1200,learning_rate=0.03,num_leaves=63,class_weight="balanced",random_state=SEED,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        m=lgb.LGBMClassifier(**lp)
        m.fit(aug_X[tr],aug_y[tr],eval_set=[(aug_X[va],aug_y[va])],callbacks=[lgb.early_stopping(100,verbose=False)])
        fp+=m.predict_proba(X_full_test)/3; del m
        m=HistGradientBoostingClassifier(max_iter=1200,learning_rate=0.03,max_leaf_nodes=63,class_weight="balanced",random_state=SEED,early_stopping=True,validation_fraction=0.1,n_iter_no_change=100)
        m.fit(aug_X[tr],aug_y[tr],sample_weight=sw)
        fp+=m.predict_proba(X_full_test)/3; del m
        aug_tst+=fp/N_FOLDS; print(f"  Aug fold {fold+1} done")
    final_probs=0.7*best_probs+0.3*aug_tst; best_name+=" + PseudoLabel"; print("Applied PL 70/30")
else:
    final_probs=best_probs

In [ ]:
preds=final_probs.argmax(1)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].astype("int64"), TARGET_COL:le.inverse_transform(preds)})
sub.to_csv("submission.csv", index=False)
print("="*60+"\nN18 RESULTS\n"+"="*60)
print(f"GBDT 3x3 ensemble OOF: {score_g:.5f}")
print(f"  solo CatBoost   OOF: {score_solo_cb:.5f}")
print(f"  solo LightGBM   OOF: {score_solo_lgb:.5f}")
print(f"  solo HGBC       OOF: {score_solo_hgbc:.5f}")
print(f"Dilution hypothesis: {'CONFIRMED' if DILUTION_CONFIRMED else 'REJECTED'} "
      f"(best solo {best_solo_name}={best_solo_score:.5f} vs ensemble={score_g:.5f})")
if score_t>0: print(f"TabPFN OOF:            {score_t:.5f}")
if score_i>0: print(f"TabICL OOF:            {score_i:.5f}")
if score_m>0: print(f"TabM OOF:              {score_m:.5f}")
print(f"Selected: {best_name} | {best_score:.5f}")
print("Priors: N17 LB=0.95034 (OOF 0.95038) | N14v3 LB=0.95029 | N06=0.95011 | N14v2=0.95002")
print(f"Delta vs N17 OOF (0.95038): {best_score-0.95038:+.5f}")
print(sub[TARGET_COL].value_counts())